In [ ]:
!pip install ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 10.0 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


# Pregunta 1

In [ ]:
from ortools.sat.python import cp_model

# 1. Modelo
model = cp_model.CpModel()

# Variables y Dominios
m1 = model.NewIntVar(1, 3, 'M1')
m2 = model.NewIntVar(1, 3, 'M2')
m3 = model.NewIntVar(1, 3, 'M3')

# Restricciones
model.AddAllDifferent([m1, m2, m3])
model.Add(m1 != 3)
model.Add(m2 != 1)
model.Add(m1 < m3)
model.Add(m3 < m2)

# Solver
solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print(f"Solución: M1={solver.Value(m1)} (Mañana), M2={solver.Value(m2)} (Noche), M3={solver.Value(m3)} (Tarde)")
    print(f"Tiempo de ejecución: {solver.WallTime()} segundos")

Solución: M1=1 (Mañana), M2=3 (Noche), M3=2 (Tarde)
Tiempo de ejecución: 0.005050285000000001 segundos


# Pregunta 2

In [ ]:
# trivial

# Pregunta 3

In [ ]:
from ortools.sat.python import cp_model

# Inicializar modelo
model = cp_model.CpModel()

# Parámetros del problema
horizon = 10     # Días del 0 al 10 (10 días de capacidad, indexados 0 a 9)
capacity = 5     # Límite máximo de obreros

durations = [2, 3, 1]  # Duraciones de Tarea 1, 2 y 3
demands = [3, 2, 4]    # Obreros necesarios para Tarea 1, 2 y 3
num_tasks = 3

# 1. Variables: start_x[t][i] es True (1) si la Tarea i inicia EXACTAMENTE en el día t
start_x = {}
for i in range(num_tasks):
    for t in range(horizon):
        start_x[t, i] = model.NewBoolVar(f'start_tarea_{i}_dia_{t}')

# 2. Restricciones de inicio y fin dentro del horizonte
for i in range(num_tasks):
    valid_starts = []
    for t in range(horizon):
        # Una tarea solo puede iniciar si tiene días suficientes para terminar antes del horizonte
        if t + durations[i] <= horizon:
            valid_starts.append(start_x[t, i])
        else:
            # Si inicia aquí, sobrepasaría el horizonte, así que lo forzamos a 0
            model.Add(start_x[t, i] == 0)

    # Cada tarea debe iniciar exactamente una vez
    model.AddExactlyOne(valid_starts)

# 3. Restricción manual de recursos (Reemplazo de AddCumulative)
# Para cada día 't', la suma de obreros de las tareas activas no debe superar la capacidad (5)
for t in range(horizon):
    obreros_activos_dia_t = []

    for i in range(num_tasks):
        tareas_empezadas = []
        # La tarea 'i' está activa en el día 't' si inició en algún día 't_prime' tal que:
        # t_prime <= t (ya empezó)  Y  t_prime > t - duracion (aún no termina)
        for t_prime in range(max(0, t - durations[i] + 1), t + 1):
            tareas_empezadas.append(start_x[t_prime, i])

        # Si la lista tiene booleanos, multiplicamos su suma (que será 0 o 1) por la demanda
        if tareas_empezadas:
            obreros_activos_dia_t.append(sum(tareas_empezadas) * demands[i])

    # Agregamos la restricción de capacidad al modelo para ese día específico
    if obreros_activos_dia_t:
        model.Add(sum(obreros_activos_dia_t) <= capacity)

# --- Variables auxiliares enteras para poder imprimir fácilmente los resultados ---
start_vars = []
end_vars = []
for i in range(num_tasks):
    s = model.NewIntVar(0, horizon, f'start_t{i+1}')
    e = model.NewIntVar(0, horizon, f'end_t{i+1}')
    # El día de inicio real es la suma de (t * booleano_de_inicio_en_t)
    model.Add(s == sum(t * start_x[t, i] for t in range(horizon)))
    model.Add(e == s + durations[i])
    start_vars.append(s)
    end_vars.append(e)

# 4. Solucionador (Solver) e impresión de resultados
solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print("¡Solución encontrada!\n")
    for i in range(num_tasks):
        inicio = solver.Value(start_vars[i])
        fin = solver.Value(end_vars[i])
        print(f"Tarea {i+1} (Requiere {demands[i]} obreros): Inicia en día {inicio}, Termina en día {fin}")

    print(f"\nTiempo de ejecución: {solver.WallTime():.4f} segundos")
else:
    print("No se encontró solución factible que cumpla con el límite de obreros.")

¡Solución encontrada!

Tarea 1 (Requiere 3 obreros): Inicia en día 8, Termina en día 10
Tarea 2 (Requiere 2 obreros): Inicia en día 5, Termina en día 8
Tarea 3 (Requiere 4 obreros): Inicia en día 0, Termina en día 1

Tiempo de ejecución: 0.0088 segundos


# Pregunta 4

In [ ]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

PA = model.NewIntVar(1, 5, 'PA')
PB = model.NewIntVar(1, 5, 'PB')

# Variables booleanas explícitas para antecedentes
b1 = model.NewBoolVar('b1')
b2 = model.NewBoolVar('b2')
b3 = model.NewBoolVar('b3')

# Reificación Bidireccional: Vincula estrictamente el booleano al estado de la condición
model.Add(PA == 3).OnlyEnforceIf(b1)
model.Add(PA != 3).OnlyEnforceIf(b1.Not())

model.Add(PA == 4).OnlyEnforceIf(b2)
model.Add(PA != 4).OnlyEnforceIf(b2.Not())

model.Add(PB == 1).OnlyEnforceIf(b3)
model.Add(PB != 1).OnlyEnforceIf(b3.Not())

# Implicaciones condicionales a los consecuentes (Si antecedente es True -> Consecuencia)
model.Add(PB == 4).OnlyEnforceIf(b1)
model.Add(PB == 5).OnlyEnforceIf(b2)
model.Add(PA == 5).OnlyEnforceIf(b3)